In [1]:
import sys
import os
import torch
import datetime as dt
import IPython.display as ipd

project_root = "/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import matcha
print("✅ Loading Matcha-TTS from:", matcha.__file__)

from matcha.models.matcha_tts import MatchaTTS
from matcha.hifigan.models import Generator as HiFiGAN
from matcha.hifigan.denoiser import Denoiser
from matcha.hifigan.config import v1
from matcha.utils.utils import intersperse
from matcha.text.symbols import symbols
from matcha.text import text_to_sequence

✅ Loading Matcha-TTS from: /rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/matcha/__init__.py


## Model Loading

In [3]:
CHECKPOINT_PATH = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/logs/multi/checkpoints/last.ckpt"

def load_matcha(path, name, vocab_size):
    print(f"📦 Loading {name} (Vocab: {vocab_size})...")
    m = MatchaTTS.load_from_checkpoint(
        path, 
        map_location="cuda", 
        strict=False,
        n_vocab=vocab_size,
        n_spks=3, 
        spk_emb_dim=64,
        weights_only=False
    )
    m.mel_mean = torch.full((1, 80, 1), -0.842103) 
    m.mel_std = torch.full((1, 80, 1), 1.573202)
    m.eval()
    return m

model_aphasia = load_matcha(CHECKPOINT_PATH, "Multi Model", 198)

print("🚀 Models loaded and ready for testing!")

📦 Loading Multi Model (Vocab: 198)...
🚀 Models loaded and ready for testing!


In [5]:
# Use the checkpoint you have in your local folder
HIFIGAN_CHECKPOINT = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/checkpoints/hifigan_T2_v1"

class AttrDict(dict):
    def __init__(self, *args, **kwargs):
        super(AttrDict, self).__init__(*args, **kwargs)
        self.__dict__ = self

def load_vocoder(checkpoint_path):
    h = AttrDict(v1)
    device = torch.device("cuda")
    hifigan = HiFiGAN(h).to(device)
    # We load the 'generator' key from the checkpoint
    state_dict = torch.load(checkpoint_path, map_location=device)
    if 'generator' in state_dict:
        hifigan.load_state_dict(state_dict['generator'])
    else:
        hifigan.load_state_dict(state_dict)
    hifigan.eval()
    hifigan.remove_weight_norm()
    return hifigan

vocoder = load_vocoder(HIFIGAN_CHECKPOINT)
denoiser = Denoiser(vocoder)
print("🎸 Local Vocoder + Denoiser Loaded!")

Removing weight norm...
🎸 Local Vocoder + Denoiser Loaded!


## Support Function

In [6]:
# --- 1. PRODUCING THE TEXT INPUTS ---
@torch.inference_mode()
def process_text(text: str):
    # 'intersperse' adds blank tokens (0) between phonemes for better rhythm
    seq = text_to_sequence(text, ['english_cleaners2'])[0]
    x = torch.tensor(intersperse(seq, 0), dtype=torch.long)[None] 
    
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long)
    
    # We'll use the symbols to see what it's saying
    x_phones = "".join([symbols[i] for i in x.squeeze(0).tolist() if i < len(symbols)])
    
    return {
        'x_orig': text,
        'x': x,
        'x_lengths': x_lengths,
        'x_phones': x_phones
    }

# --- 2. THE WAVEFORM CONVERTER ---
# Adding a simple denoiser logic since HiFi-GAN can be 'hissy'
def to_waveform(mel, vocoder):
    with torch.no_grad():
        # Ensure mel is [1, 80, T]
        if mel.ndim == 4: mel = mel.squeeze(1)
        
        audio = vocoder(mel).clamp(-1, 1)
        
        # Note: If your local vocoder doesn't have a built-in denoiser, 
        # we skip the 'strength' part to avoid errors, or use a simple squeeze.
        return audio.cpu().squeeze()

# --- 3. THE SYNTHESIS ENGINE ---
@torch.inference_mode()
def synthesise(text, model, n_timesteps=10, temperature=0.667, length_scale=1.0):
    text_processed = process_text(text)
    start_t = dt.datetime.now()
    
    output = model.synthesise(
        text_processed['x'],
        text_processed['x_lengths'],
        n_timesteps=n_timesteps,
        temperature=temperature,
        spks=None,
        length_scale=length_scale
    )
    
    output.update({'start_t': start_t, **text_processed})
    return output

# --- 4. THE VOCODER PROCESS ---
@torch.inference_mode()
def to_waveform(mel, vocoder, denoiser):
    # Fix the 4D broadcasting error [1, 80, 80, T] -> [1, 80, T]
    if mel.ndim == 4:
        if mel.shape[1] == mel.shape[2] == 80:
            mel = mel[:, 0, :, :] 
        else:
            mel = mel.squeeze(1)

    # Convert to audio
    audio = vocoder(mel).clamp(-1, 1)
    
    # Apply Denoiser (This stops the 'Robotic' metallic sound)
    audio = denoiser(audio.squeeze(0), strength=0.00025).cpu().squeeze()
    
    return audio

# --- 5. RUNNING TEXT-TO-SPEECH ---
def run_synthesis(text, model_to_use, name, id=0, n=15, temp=0.5, length=1.0):
    print(f"🎬 Testing: {name}")
    
    # 1. Process Text (Intersperse adds the 'breathing room' between phonemes)
    device = model_to_use.device
    seq = intersperse(text_to_sequence(text, ['english_cleaners2'])[0], 0)
    x = torch.tensor(seq, dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=device)
    speaker_id = torch.tensor([id], dtype=torch.long, device=device)
    
    # 2. Synthesise Mel
    with torch.no_grad():
        output = model_to_use.synthesise(
            x, x_lengths, 
            n_timesteps=n, 
            temperature=temp, 
            length_scale=length,
            spks=speaker_id
        )
        
        # 3. Waveform + Denoise
        audio = to_waveform(output['mel'], vocoder, denoiser)
        
    display(ipd.Audio(audio, rate=22050))

## Running Text-to-Speech

In [13]:
# Run the test
run_synthesis("But I um. I found... seven... in seventy.... sixty-four, married. eighty-two.", model_aphasia, "Aphasia Voice", 0, n=100, temp=0.8)
run_synthesis("I uh, am absolutely uhhh. finished with uh... these architecture errors!", model_aphasia, "Dementia Voice", 1)
run_synthesis("I uh, am absolutely uhhh. finished with uh... these architecture errors!", model_aphasia, "ASD Voice", 2)

🎬 Testing: Aphasia Voice


🎬 Testing: Dementia Voice


🎬 Testing: ASD Voice


## Script Manipulation

In [ ]:
import spacy
import random
import re

# Load the microscopic English NLP model (takes < 1 second)
nlp = spacy.load("en_core_web_sm")

In [ ]:
def generate_false_start(word):
    """
    Creates progressive phonetic false starts.
    Example: 'hospital' -> 'ho... hosp... hospital'
    """
    # Strip whitespace and punctuation to get the raw word
    clean_word = re.sub(r'[^\w\s]', '', word)
    
    # If the word is tiny (like "my" or "is"), just stutter the first letter
    if len(clean_word) <= 2:
        return f"{clean_word[0]}... {word}"
        
    stutters = []
    # Weighted random choice: 70% chance of 1 stumble, 30% chance of 2 stumbles
    num_stutters = random.choices([1, 2], weights=[0.7, 0.3])[0]
    
    if num_stutters == 1:
        # Cut the word somewhere in the first half
        cut = random.randint(1, max(1, len(clean_word) - 2))
        stutters.append(f"{clean_word[:cut]}...")
    else:
        # Progressive stutter: cut early, then cut a bit later
        cut1 = max(1, len(clean_word) // 3)
        cut2 = max(cut1 + 1, int(len(clean_word) * 0.75))
        stutters.append(f"{clean_word[:cut1]}...")
        if cut2 < len(clean_word):
            stutters.append(f"{clean_word[:cut2]}...")
            
    # Reattach the trailing whitespace/punctuation from the original word
    return " ".join(stutters) + f" {word}"

def inject_aphasia_cues(text, severity=0.3):
    """Upgraded injector with fillers, repetitions, and false starts."""
    doc = nlp(text)
    fillers = ["um...", "uh,", "uuum...", "..."]
    
    manipulated_words = []
    
    for token in doc:
        is_content_word = token.pos_ in ["NOUN", "VERB", "ADJ", "PROPN"]
        
        if is_content_word and random.random() < severity:
            
            # Now we have 3 clinically accurate dysfluency types!
            stutter_type = random.choices(
                ["filler", "repetition", "false_start"], 
                weights=[0.4, 0.2, 0.4] # 40% filler, 20% repeat word, 40% false start
            )[0]
            
            if stutter_type == "false_start":
                # Inject the progressive stumble and SKIP adding the normal word
                manipulated_words.append(generate_false_start(token.text_with_ws))
                continue 
                
            elif stutter_type == "repetition" and token.i > 0 and doc[token.i - 1].pos_ != "PUNCT":
                prev_word = doc[token.i - 1].text
                manipulated_words.append(f"... {prev_word} ")
                
            elif stutter_type == "filler":
                filler = random.choice(fillers)
                manipulated_words.append(f"{filler} ")
                
        manipulated_words.append(token.text_with_ws)
        
    final_text = "".join(manipulated_words)
    final_text = re.sub(r'\s+([?.!,])', r'\1', final_text)
    return re.sub(r'\s{2,}', ' ', final_text).strip()

In [ ]:
# --- LET'S TEST IT ---
clean_script = "Hello doctor, I am feeling a bit dizzy today and my chest hurts."

# Mild Aphasia
print("Mild:  ", inject_aphasia_cues(clean_script, severity=0.15))

# Severe Aphasia
print("Severe:", inject_aphasia_cues(clean_script, severity=0.6))

## Full Inference Pipeline

In [ ]:
sentence = "Hello doctor, I am feeling a bit dizzy today and my chest hurts."

manipulated_sentence = inject_aphasia_cues(sentence, severity=0.7)
run_synthesis(manipulated_sentence, model_aphasia, "Aphasia Voice", 0)

In [ ]:
manipulated_sentence

## Accent Vector Test (VCTK Voice vs VCTK-Indian)

In [ ]:
CHECKPOINT_PATH = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/checkpoints/matcha_vctk.ckpt"

def load_matcha(path, name, vocab_size):
    print(f"📦 Loading {name} (Vocab: {vocab_size})...")
    m = MatchaTTS.load_from_checkpoint(
        path, 
        map_location="cuda", 
        strict=False,
        n_vocab=vocab_size,
        n_spks=109, 
        spk_emb_dim=64,
        weights_only=False
    )
    m.mel_mean = torch.full((1, 80, 1), -6.630575) 
    m.mel_std = torch.full((1, 80, 1), 2.482914)
    m.eval()
    return m

model_vctk = load_matcha(CHECKPOINT_PATH, "Multi Model", 178)

print("🚀 Models loaded and ready for testing!")

In [ ]:
def run_synthesis_base(text, model_to_use, name, id=0):
    print(f"🎬 Testing: {name}")
    
    # 1. Process Text (Intersperse adds the 'breathing room' between phonemes)
    device = model_to_use.device
    seq = intersperse(text_to_sequence(text, ['english_cleaners2'])[0], 0)
    x = torch.tensor(seq, dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=device)
    speaker_id = torch.tensor([id], dtype=torch.long, device=device)
    
    # 2. Synthesise Mel
    with torch.no_grad():
        output = model_to_use.synthesise(
            x, x_lengths, 
            n_timesteps=50, 
            temperature=0.5, 
            length_scale=1.35,
            spks=speaker_id
        )
        
        # 3. Waveform + Denoise
        audio = to_waveform(output['mel'], vocoder, denoiser)
        
    display(ipd.Audio(audio, rate=22050))

# Run the test
run_synthesis_base("This is a test of accent to check if it actually sounds legit", model_vctk, "Base Model", 3)

In [ ]:
accent_vector = torch.load("/rds/general/user/ak8224/home/MedEmoji-TTS/vector/indian_accent_vector.pt")

def run_synthesis_vector(text, model_to_use, name, id=0, shift=None, alpha=0.0):
    print(f"🎬 Testing: {name} (Alpha: {alpha})")
    
    device = model_to_use.device
    seq = intersperse(text_to_sequence(text, ['english_cleaners2'])[0], 0)
    x = torch.tensor(seq, dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=device)
    
    # Let's test this on Speaker 0 (Your Aphasia Patient)
    speaker_id = torch.tensor([id], dtype=torch.long, device=device) 
    
    with torch.no_grad():
        output = model_to_use.synthesise(
            x, x_lengths, 
            n_timesteps=50, 
            temperature=0.667, 
            length_scale=1.0,
            spks=speaker_id,
            latent_shift=shift,
            alpha=alpha
        )
        
        audio = to_waveform(output['mel'], vocoder, denoiser)
        
    display(ipd.Audio(audio, rate=22050))


# 1. Normal
run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_vctk, "Baseline", 0, shift=None)

# 2. Normal + Indian Accent Vector
run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_vctk, "Accent Shift", 0, shift=accent_vector, alpha=1.0)

# 3. Normal + Indian Accent Vector (Overcranked)
run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_vctk, "Extreme Accent", 0, shift=accent_vector, alpha=2.0)

## Fine-tune Voice Testing

In [ ]:
CHECKPOINT_PATH = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/logs/accent/checkpoints/checkpoint_epoch=499.ckpt"

def load_matcha(path, name, vocab_size):
    print(f"📦 Loading {name} (Vocab: {vocab_size})...")
    m = MatchaTTS.load_from_checkpoint(
        path, 
        map_location="cuda", 
        strict=False,
        n_vocab=vocab_size,
        n_spks=14, 
        spk_emb_dim=64,
        weights_only=False
    )
    m.mel_mean = torch.full((1, 80, 1), -0.669601)
    m.mel_std = torch.full((1, 80, 1), 1.644132)
    m.eval()
    return m

model_accent = load_matcha(CHECKPOINT_PATH, "Multi Accent Model", 198)

print("🚀 Models loaded and ready for testing!")

In [ ]:
test_text = "This is a test of accent to check if it actually sounds legit"

# Female Sound
run_synthesis(test_text, model_accent, "Arabic Accent", id=0, n=50, temp=0.3, length=0.95)
run_synthesis(test_text, model_accent, "Vietnamese Accent", id=2, n=50, temp=0.3, length=0.95)
run_synthesis(test_text, model_accent, "Korean Accent", id=4, n=50, temp=0.3, length=0.95)
run_synthesis(test_text, model_accent, "Spanish Accent", id=6, n=50, temp=0.3, length=0.95)
run_synthesis(test_text, model_accent, "Indian Accent", id=8, n=50, temp=0.3, length=0.95)
run_synthesis(test_text, model_accent, "Mandarin Accent", id=10, n=50, temp=0.3, length=0.95)
run_synthesis(test_text, model_accent, "Base Voice (VCTK)", id=12, n=50, temp=0.3, length=0.95)

In [ ]:
test_text = "This is a test of accent to check if it actually sounds legit"

# Male Sound
run_synthesis(test_text, model_accent, "Arabic Accent", id=1, n=50, temp=0.6, length=0.95)
run_synthesis(test_text, model_accent, "Vietnamese Accent", id=3, n=50, temp=0.6, length=0.95)
run_synthesis(test_text, model_accent, "Korean Accent", id=5, n=50, temp=0.6, length=0.95)
run_synthesis(test_text, model_accent, "Spanish Accent", id=7, n=50, temp=0.6, length=0.95)
run_synthesis(test_text, model_accent, "Indian Accent", id=9, n=50, temp=0.6, length=0.95)
run_synthesis(test_text, model_accent, "Mandarin Accent", id=11, n=50, temp=0.6, length=0.95)
run_synthesis(test_text, model_accent, "Base Voice (VCTK)", id=13, n=50, temp=0.6, length=0.95)

## Aphasia with Vector Shift

In [ ]:
CHECKPOINT_PATH = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/logs/multi/checkpoints/last.ckpt"

def load_matcha(path, name, vocab_size):
    print(f"📦 Loading {name} (Vocab: {vocab_size})...")
    m = MatchaTTS.load_from_checkpoint(
        path, 
        map_location="cuda", 
        strict=False,
        n_vocab=vocab_size,
        n_spks=3, 
        spk_emb_dim=64,
        weights_only=False
    )
    m.mel_mean = torch.full((1, 80, 1), -0.842103) 
    m.mel_std = torch.full((1, 80, 1), 1.573202)
    m.eval()
    return m

model_medemoji = load_matcha(CHECKPOINT_PATH, "Multi Accent Model", 198)

print("🚀 Models loaded and ready for testing!")
print("🧮 Loading Latent Accent Vectors...")

def safe_load_vector(path):
    try:
        return torch.load(path, map_location="cuda")
    except FileNotFoundError:
        print(f"⚠️ Vector not found at {path}. Will default to Base.")
        return None

vector_dir = "/rds/general/user/ak8224/home/MedEmoji-TTS/vector/final"

accent_vectors = {
    "Base": None,
    "Hindi": safe_load_vector(f"{vector_dir}/indian_accent_vector.pt"),
    "Mandarin": safe_load_vector(f"{vector_dir}/mandarin_accent_vector.pt"),
    "Korean": safe_load_vector(f"{vector_dir}/korean_accent_vector.pt"),
    "Vietnamese": safe_load_vector(f"{vector_dir}/vietnamese_accent_vector.pt"),
    "Arabic": safe_load_vector(f"{vector_dir}/arabic_accent_vector.pt"),
    "Spanish": safe_load_vector(f"{vector_dir}/spanish_accent_vector.pt")
}

print("🚀 All accent vectors loaded!")

In [ ]:
def run_synthesis_vector(text, model_to_use, name, id=0, shift=None, alpha=0.0):
    print(f"🎬 Testing: {name} (Alpha: {alpha})")
    
    device = model_to_use.device
    seq = intersperse(text_to_sequence(text, ['english_cleaners2'])[0], 0)
    x = torch.tensor(seq, dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=device)
    
    speaker_id = torch.tensor([id], dtype=torch.long, device=device) 
    
    # 🚨 THE FIX: Bypass Autograd using .data
    if shift is not None:
        # 1. Save the original speaker identity
        original_embedding = model_to_use.spk_emb.weight.data[id].clone()
        
        # 2. Inject the accent vector directly into the raw tensor data
        model_to_use.spk_emb.weight.data[id] = original_embedding + (alpha * shift.to(device))
    
    with torch.no_grad():
        output = model_to_use.synthesise(
            x, x_lengths, 
            n_timesteps=50, 
            temperature=0.667, 
            length_scale=1.0,
            spks=speaker_id
        )
        
    if shift is not None:
        model_to_use.spk_emb.weight.data[id] = original_embedding
        
    audio = to_waveform(output['mel'], vocoder, denoiser)
    display(ipd.Audio(audio, rate=22050))

sentences = "Hello doctor, I am feeling a bit dizzy."

# 1. Normal
run_synthesis_vector(sentences, model_medemoji, "Baseline", 0, shift=None)

run_synthesis_vector(sentences, model_medemoji, "Hindi Shift", 0, shift=accent_vectors["Hindi"], alpha=5.0)

run_synthesis_vector(sentences, model_medemoji, "Vietnamese Shift", 0, shift=accent_vectors["Vietnamese"], alpha=5.0)

run_synthesis_vector(sentences, model_medemoji, "Mandarin Shift", 0, shift=accent_vectors["Mandarin"], alpha=5.0)

run_synthesis_vector(sentences, model_medemoji, "Spanish Shift", 0, shift=accent_vectors["Spanish"], alpha=5.0)

run_synthesis_vector(sentences, model_medemoji, "Arabic Shift", 0, shift=accent_vectors["Arabic"], alpha=5.0)

run_synthesis_vector(sentences, model_medemoji, "Korean Shift", 0, shift=accent_vectors["Korean"], alpha=5.0)

In [ ]:
def run_synthesis_vector(text, model_to_use, name, id=0, shift=None, alpha=0.0):
    print(f"🎬 Testing: {name} (Alpha: {alpha})")
    
    device = model_to_use.device
    seq = intersperse(text_to_sequence(text, ['english_cleaners2'])[0], 0)
    x = torch.tensor(seq, dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=device)
    
    speaker_id = torch.tensor([id], dtype=torch.long, device=device) 
    
    # 🚨 THE FIX: Bypass Autograd using .data
    if shift is not None:
        # 1. Save the original speaker identity
        original_embedding = model_to_use.spk_emb.weight.data[id].clone()
        
        # 2. Inject the accent vector directly into the raw tensor data
        model_to_use.spk_emb.weight.data[id] = original_embedding + (alpha * shift.to(device))
    
    with torch.no_grad():
        output = model_to_use.synthesise(
            x, x_lengths, 
            n_timesteps=50, 
            temperature=0.667, 
            length_scale=1.0,
            spks=speaker_id
        )
        
    # 🚨 CLEANUP: Restore the original identity using .data
    if shift is not None:
        model_to_use.spk_emb.weight.data[id] = original_embedding
        
    audio = to_waveform(output['mel'], vocoder, denoiser)
    display(ipd.Audio(audio, rate=22050))


# 1. Normal
run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_medemoji, "Baseline", 2, shift=None)

run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_medemoji, "Hindi Shift", 2, shift=accent_vectors["Hindi"], alpha=5.0)

run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_medemoji, "Vietnamese Shift", 2, shift=accent_vectors["Vietnamese"], alpha=5.0)

run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_medemoji, "Mandarin Shift", 2, shift=accent_vectors["Mandarin"], alpha=5.0)

run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_medemoji, "Spanish Shift", 2, shift=accent_vectors["Spanish"], alpha=5.0)

run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_medemoji, "Arabic Shift", 2, shift=accent_vectors["Arabic"], alpha=5.0)

run_synthesis_vector("Hello doctor, I am feeling a bit dizzy.", model_medemoji, "Korean Shift", 2, shift=accent_vectors["Korean"], alpha=5.0)